In [ ]:
from huggingface_hub import login
import os
import sys
import csv
from tqdm import trange
from transformers import AutoModel,AutoTokenizer
FILE_PATH = './QA_results_GT.csv'
os.environ["OPENAI_API_KEY"] = AAA

In [ ]:
# 定义分析文件路径，指向包含不同模型回答的CSV文件
ANA_FILE_PATH = './mthp_output.csv'

# 初始化存储不同模型回答的列表
naiveanswer_LIST = []  # 存储naive模型的回答
lightraganswer_LIST = []  # 存储lightrag模型的回答
minianswer_LIST = []  # 存储minirag模型的回答
QUESTION_LIST = []  # 存储问题列表
GA_LIST = []  # 存储标准答案列表
filelength = 0  # 初始化文件行数计数器

# 打开CSV文件并读取数据
with open(ANA_FILE_PATH, mode='r', encoding='utf-8') as question_file:
    # 创建CSV字典读取器
    reader = csv.DictReader(question_file)
    # 遍历每一行数据
    for row in reader:
        # 将问题添加到问题列表
        QUESTION_LIST.append(row['Question'])
        # 将标准答案添加到标准答案列表
        GA_LIST.append(row['Gold Answer'])
        # 将naive模型的回答添加到对应列表
        naiveanswer_LIST.append(row['naive'])
        # 将lightrag模型的回答添加到对应列表
        lightraganswer_LIST.append(row['lightrag'])
        # 将minirag模型的回答添加到对应列表
        minianswer_LIST.append(row['minirag'])
        # 增加文件行数计数
        filelength = filelength+1

In [ ]:
# 定义评估提示模板，用于LLM评估不同模型回答的质量
PROMPT = """
# 任务说明：评估三个不同学生对同一问题的回答质量
现在，我将给你一个问题、该问题的标准答案，以及三个不同学生提供的答案。

# 评分规则定义
根据以下规则判断答案：
- 如果答案正确，得1分。
- 如果答案与问题无关，得0分。
- 如果答案不正确，得-1分。

# 输出格式要求
请以JSON格式返回你的评分结果。

# 示例说明
例如：

问题：
李华什么时候到达城市？

标准答案：
20260105

答案1：李华于1月5日下午到达
答案2：抱歉，您提供的信息中没有关于李华到达的信息
答案3：您提供的信息中没有准确答案，但根据找到的第一条信息，李华于4月17日到达

输出：
{{
"Score1": 1,  // 答案1正确
"Score2": 0,  // 答案2无关
"Score3": -1, // 答案3错误
}}



# 实际数据
现在开始评估实际数据：

问题：
{question}  // 占位符，将替换为实际问题
标准答案：
{ga}  // 占位符，将替换为实际标准答案

答案1：{naive}  // 占位符，将替换为naive模型的回答
答案2：{light}  // 占位符，将替换为lightrag模型的回答
答案3：{mini}  // 占位符，将替换为minirag模型的回答

输出：

"""

In [ ]:
PROMPT = """
Now, I'll give you a question, a gold answer to this question, and three answers provided by different students.

Determine the answer according to the following rules:
If the answer is correct, get 1 point.
If the answer is irrelevant to the question, it will receive 0 points.
If the answer is incorrect, get -1 point.

Return your answer in JSON mode.

For example:

Question:
When does Li Hua arrive to the city?

Gold Answer:
20260105

Answer1: LiHua arrived on the afternoon of January 5th
Answer2: Sorry, there is no information about LiHua's arrival in the information you provided
Answer3: There is no accurate answer in the information you provided, but according to the first information found, LiHua arrived on April 17th

output:
{{
"Score1": 1,
"Score2": 0,
"Score3": -1,
}}



Real data:

Question:
{question}
Gold Answer:
{ga}

Answer1: {naive}
Answer2: {light}
Answer3: {mini}

output:

"""

In [ ]:
# DeepSeek模型评估部分
# 导入OpenAI客户端库
from openai import OpenAI
# 初始化DeepSeek聊天客户端，配置API密钥和基础URL
chatbot = OpenAI(api_key=My_deepseek_key, base_url="https://api.deepseek.com")

# 初始化存储聊天响应的列表
chat_list = []
# 遍历所有问题（按文件长度）
for i in range(filelength):
    # 格式化提示词，替换占位符为实际问题和答案
    p = PROMPT.format(
        question=QUESTION_LIST[i],  # 当前问题
        ga=GA_LIST[i],  # 标准答案
        naive=naiveanswer_LIST[i],  # naive模型回答
        light=lightraganswer_LIST[i],  # lightrag模型回答
        mini=minianswer_LIST[i]  # minirag模型回答
    )
    # 调用DeepSeek API进行评估
    chat_completion = chatbot.chat.completions.create(
        messages=[
            {
                "role": "system",  # 设置消息角色为系统
                "content": p,  # 提示词内容
            },
        ],
        model="deepseek-chat",  # 使用deepseek-chat模型
        stream=False  # 非流式响应
    )
    # 提取并存储回答内容，去除首尾空白字符
    chat_list.append(chat_completion.choices[0].message.content.strip())

In [ ]:
#openai
from openai import OpenAI
from tqdm import trange
chatbot = OpenAI()
chat_list = []
for i in trange(filelength):
    p = PROMPT.format(question = QUESTION_LIST[i], ga = GA_LIST[i], naive = naiveanswer_LIST[i], light = lightraganswer_LIST[i], mini = minianswer_LIST[i])
    chat_completion = chatbot.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content":p,
            },
        ],
        model="gpt-4o",
    )
    chat_list.append(chat_completion.choices[0].message.content.strip())


In [ ]:
# 导入JSON处理模块
import json
# 导入json_repair模块，用于修复格式不完整的JSON
import json_repair

# 初始化存储评分结果的列表
chat_score_list = []    

# 遍历所有聊天响应
for chat in chat_list:
    try:
        # 修复JSON并解析，去除可能的Markdown代码块标记
        data = json_repair.loads(chat.strip('```json').strip('```'))
        # 添加到评分列表
        chat_score_list.append(data)
    except:
        # 如果解析失败，添加0作为错误标记
        chat_score_list.append(0)
        # 打印错误信息
        print('Error in chat:', chat)

# 提取各模型的评分列表
all_score1 = [data['Score1'] for data in chat_score_list]  # naive模型评分
all_score2 = [data['Score2'] for data in chat_score_list]  # lightrag模型评分
all_score3 = [data['Score3'] for data in chat_score_list]  # minirag模型评分

# 统计naive模型的评分分布
all_score1_1 = all_score1.count(1)  # 正确回答数量
all_score1_0 = all_score1.count(0)  # 无关回答数量
all_score1_neg = all_score1.count(-1)  # 错误回答数量

# 统计lightrag模型的评分分布
all_score2_1 = all_score2.count(1)  # 正确回答数量
all_score2_0 = all_score2.count(0)  # 无关回答数量
all_score2_neg = all_score2.count(-1)  # 错误回答数量

# 统计minirag模型的评分分布
all_score3_1 = all_score3.count(1)  # 正确回答数量
all_score3_0 = all_score3.count(0)  # 无关回答数量
all_score3_neg = all_score3.count(-1)  # 错误回答数量

# 计算总样本数
all = len(all_score1)

# 打印各模型的评分分布（数量）
print(all_score1_1, all_score1_0, all_score1_neg)  # naive模型评分数量
print(all_score2_1, all_score2_0, all_score2_neg)  # lightrag模型评分数量
print(all_score3_1, all_score3_0, all_score3_neg)  # minirag模型评分数量

# 打印各模型的评分分布（百分比）
print(f"Score1(naive) 正确: {all_score1_1 / all * 100:.2f}%, 无关: {all_score1_0 / all * 100:.2f}%, 错误: {all_score1_neg / all * 100:.2f}%")    
print(f"Score2(lightrag) 正确: {all_score2_1 / all * 100:.2f}%, 无关: {all_score2_0 / all * 100:.2f}%, 错误: {all_score2_neg / all * 100:.2f}%")
print(f"Score3(minirag) 正确: {all_score3_1 / all * 100:.2f}%, 无关: {all_score3_0 / all * 100:.2f}%, 错误: {all_score3_neg / all * 100:.2f}%")